In [9]:
import sys
sys.executable

'/home/gianl/miniconda3/envs/vm2/bin/python'

In [10]:
import os
os.getcwd()


'/home/gianl/Dokumente/projekte/VM2-RC-Modell-ui/notebooks'

In [11]:
import os, sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
print("PROJECT_ROOT =", PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

PROJECT_ROOT = /home/gianl/Dokumente/projekte/VM2-RC-Modell-ui


In [12]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat
from pathlib import Path

from core.bootstrap import create_facade # imports the connection to the midlayer


In [13]:
PROJECT_ID = "rc-model-validation"
VARIANT_ID = "Val"
facade_Val = create_facade(PROJECT_ID, VARIANT_ID)

facade_Val.run_simulation(PROJECT_ID, VARIANT_ID)

Simulation finished.


RunReport(ok=True, run_id='latest', message='Simulation for project rc-model-validation and variant Val completed successfully.')

In [14]:
# load results
df_res = facade_Val._result.load_raw()
df_res.head(5)

,temperature_outdoor_air,temperature_air_room,temperature_in_glazing_north,temperature_in_glazing_east,temperature_in_glazing_south,temperature_in_glazing_west,temperature_out_glazing_north,temperature_out_glazing_east,temperature_out_glazing_south,temperature_out_glazing_west,...,temperature_wall_s_4,temperature_wall_w_4,temperature_roof_4,temperature_floor_4,temperature_int_wall_4,temperature_int_ceiling_4,output_heating_power,output_cooling_power,output_lighting_electricity,output_equipment_electricity
datetime,,,,,,,,,,,,,,,,,,,,,
2019-01-01 00:00:00,-1.443,20.999935,19.088249,19.088249,19.088249,19.088249,0.260929,0.260929,0.260929,0.260929,...,-1.680789,-1.696373,-1.693487,11.790370,21.039693,21.036238,1.909770e+07,0.0,-2.273737e-13,753.12
2019-01-01 01:00:00,-0.443,20.999938,19.164621,19.164621,19.164621,19.164621,1.172695,1.172695,1.172695,1.172695,...,-0.627774,-0.641286,-0.639342,11.787802,20.993549,21.026543,1.825297e+07,0.0,-2.273737e-13,753.12
2019-01-01 02:00:00,0.500,20.999941,19.233123,19.233123,19.233123,19.233123,1.974286,1.974286,1.974286,1.974286,...,0.327066,0.314863,0.316224,11.785233,20.954753,21.014793,1.748479e+07,0.0,-2.273737e-13,753.12
2019-01-01 03:00:00,1.329,20.999942,19.274204,19.274204,19.274204,19.274204,2.461602,2.461602,2.461602,2.461602,...,1.116091,1.104798,1.105764,11.782660,20.922465,21.001919,1.691561e+07,0.0,-2.273737e-13,753.12
2019-01-01 04:00:00,1.833,20.999942,19.283699,19.283699,19.283699,19.283699,2.596898,2.596898,2.596898,2.596898,...,1.652729,1.642131,1.642805,11.780083,20.895587,20.988577,1.664819e+07,0.0,-2.273737e-13,753.12


# load matlab result

In [15]:
mat_path = Path('projects') / 'rc-model-validation' / 'mat-reference' / 'matlab_ref_results.mat'
mat_path_2 = Path(PROJECT_ROOT) / 'projects' / 'rc-model-validation' / 'mat-reference' / 'matlab_ref_results.mat'
print(PROJECT_ROOT)
print(mat_path)
print(mat_path_2)

mat_data = loadmat(mat_path_2, squeeze_me=True)

# Extract relevant output data from MATLAB structure
Q_c_ref = mat_data['output_cooling_power']
Q_h_ref = mat_data['output_heating_power']
El_ref = mat_data['output_lighting_electricity']
Eq_ref = mat_data['output_equipment_electricity']

# Create DataFrame for MATLAB outputs
df_outputs = pd.DataFrame({
    'output_cooling_power': Q_c_ref,
    'output_heating_power': Q_h_ref,
    'output_lighting_electricity': El_ref,
    'output_equipment_electricity': Eq_ref
})


try:
    row_names_all = df_res.columns.tolist()
    row_names = row_names_all[1:-4]  # Exclude first and last four columns
except NameError:
    print("df_res is not defined.")
    
temp_mat_array = mat_data['output_temperatures']
num_rows_mat = temp_mat_array.shape[1]

if len(row_names) != num_rows_mat:
    print("Warning: Number of row names does not match number of rows in the MATLAB data.")
    print(f"Number of row names: {len(row_names)}, Number of rows in MATLAB data: {num_rows_mat}")
else:
    df_temp_outputs = pd.DataFrame(
        data=temp_mat_array,
        columns=row_names
    )

df_res_mat = pd.concat([df_temp_outputs, df_outputs], axis=1)
df_res_mat.head()


/home/gianl/Dokumente/projekte/VM2-RC-Modell-ui
projects/rc-model-validation/mat-reference/matlab_ref_results.mat
/home/gianl/Dokumente/projekte/VM2-RC-Modell-ui/projects/rc-model-validation/mat-reference/matlab_ref_results.mat


,temperature_air_room,temperature_in_glazing_north,temperature_in_glazing_east,temperature_in_glazing_south,temperature_in_glazing_west,temperature_out_glazing_north,temperature_out_glazing_east,temperature_out_glazing_south,temperature_out_glazing_west,temperature_in_frame_north,...,temperature_wall_s_4,temperature_wall_w_4,temperature_roof_4,temperature_floor_4,temperature_int_wall_4,temperature_int_ceiling_4,output_cooling_power,output_heating_power,output_lighting_electricity,output_equipment_electricity
0,20.905853,18.793906,18.793906,18.793906,18.793906,-0.713638,-0.713638,-0.713638,-0.713638,13.990463,...,-1.199977,-1.201227,-1.324237,11.744259,20.730805,20.772433,0.0,21063.686547,-2.273737e-13,753.12
1,20.908765,18.886626,18.886626,18.886626,18.886626,0.253663,0.253663,0.253663,0.253663,14.298536,...,-0.192939,-0.194127,-0.295613,11.741659,20.719788,20.765526,0.0,19995.501184,-2.273737e-13,753.12
2,20.911558,18.974430,18.974430,18.974430,18.974430,1.165840,1.165840,1.165840,1.165840,14.589330,...,0.738508,0.737377,0.641098,11.739059,20.712807,20.758921,0.0,19381.661872,-2.273737e-13,753.12
3,20.914053,19.051837,19.051837,19.051837,19.051837,1.967752,1.967752,1.967752,1.967752,14.845135,...,1.558268,1.557192,1.466090,11.736458,20.708659,20.752766,0.0,18813.129875,-2.273737e-13,753.12
4,20.915564,19.098664,19.098664,19.098664,19.098664,2.455275,2.455275,2.455275,2.455275,15.000477,...,2.060295,2.059270,1.974384,11.733857,20.706447,20.747101,0.0,18389.275320,-2.273737e-13,753.12
